# Denoising diffusion (U-Net)

A conditional DDPM: a ConvNeXt U-Net predicts the noise from $[x_t, y]$, where $y$ is concatenated as a
second channel. It was trained on the 22,000 training patches with a 1000-step cosine schedule and an L1 loss.
Weights: `weights/ddpm_conditional_unet_raw.pth`, the **raw** training weights (no EMA), like every other
learned method in Table 3.

Inference runs 500 reverse steps from Gaussian noise and draws 10 posterior samples per patch. **The 10
samples are averaged pixel-wise into the posterior mean, and the metrics are computed on that image.**

*Number of reverse steps.* The sampler starts from Gaussian noise and runs the last `steps` steps of the
1000-step training schedule. The step count was chosen on 20 training patches (every 1100th file of the
training set, seed 0): the posterior mean reaches 56.19, 56.36, 56.66 and 56.29 dB with 100, 250, 500 and
1000 steps, so 500 steps are used.
The sampler is stochastic. Its noise comes from a CPU random generator with a fixed seed, so it is the
same on every device. The Table-3 row uses seed 1; seeds 0 and 2 are printed to show the spread (0.01 dB).
About 7 minutes on an RTX 3090.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the ground-level field continued 800 m upward in a single step. This model was trained with zero padding to 64 × 64 (`expand=100`), so the same padding is used here.

In [2]:
from gravdc import upward_continue
from gravdc.sampling import ddpm_sample

names, x = load_test_set(DATA_DIR)
x = x.to(device)
y = upward_continue(x, 800, expand=100)
print(len(names), "patches:", names)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Model and inference

In [3]:
SEEDS, SEED = [0, 1, 2], 1  # the Table-3 row uses seed 1
model = load_model("ddpm_unet", device)  # raw (non-EMA) weights
samples_by_seed = {s: ddpm_sample(model, y, timesteps=1000, steps=500, n_samples=10, seed=s) for s in SEEDS}
samples = samples_by_seed[SEED]  # (10 samples, 10 patches, 1, 32, 32)

## Metrics (posterior mean of the 10 samples)

In [4]:
def posterior_mean_metrics(samples):
    """Average the 10 samples of each patch pixel-wise (posterior mean), then score that one image."""
    return evaluate(x, samples.mean(0))


seed_summaries = {s: summarize(posterior_mean_metrics(samples_by_seed[s])) for s in SEEDS}
for s, summary in seed_summaries.items():
    print(f"seed {s}:  {format_summary(summary)}")
print()
per_image = posterior_mean_metrics(samples)
compare("Denoising diffusion (U-Net)", summarize(per_image))
print("\nper-patch PSNR (seed 1):", " ".join(f"{m['PSNR']:.2f}" for m in per_image))
save_result("Denoising diffusion (U-Net)", per_image, names, RESULTS_DIR, seed=SEED, steps=500,
            aggregation="posterior mean of 10 samples",
            seed_summaries={str(s): v for s, v in seed_summaries.items()})

seed 0:  PSNR: 38.4442 ± 2.7472  SSIM: 0.9910 ± 0.0089  RE: 0.0619 ± 0.0214  RMSE: 0.0219 ± 0.0091
seed 1:  PSNR: 38.4493 ± 2.7439  SSIM: 0.9911 ± 0.0089  RE: 0.0618 ± 0.0214  RMSE: 0.0218 ± 0.0091
seed 2:  PSNR: 38.4391 ± 2.7433  SSIM: 0.9910 ± 0.0089  RE: 0.0619 ± 0.0214  RMSE: 0.0219 ± 0.0091

Denoising diffusion (U-Net)
  metric             this run              Table 3
  PSNR       38.4493 ± 2.7439     38.4493 ± 2.7439  ✓
  SSIM        0.9911 ± 0.0089      0.9911 ± 0.0089  ✓
  RE          0.0618 ± 0.0214      0.0618 ± 0.0214  ✓
  RMSE        0.0218 ± 0.0091      0.0218 ± 0.0091  ✓

per-patch PSNR (seed 1): 33.17 42.58 38.27 38.72 37.26 40.53 38.90 36.23 36.46 42.38


In [5]:
# Figures of the field data are off by default (the executed notebooks are kept small and text-only).
# Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    y_plot = upward_continue(x, 800)  # 36 x 36 padding, for the figure only
    show_reconstruction(x[k], y_plot[k], {"posterior mean (10 samples)": samples[:, k].mean(0), "sample 1": samples[0, k]},
                        title=names[k])